In [2]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import pickle
from itertools import combinations

# 데이터 수집 (확장)
import yfinance as yf
import requests
import feedparser
from bs4 import BeautifulSoup
import pandas_datareader.data as web

# 감성 분석 (고도화)
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 고급 머신러닝
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor, VotingRegressor, BaggingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# XGBoost & LightGBM & CatBoost
try:
    import xgboost as xgb
    import lightgbm as lgb
    # import catboost as cb
except ImportError:
    print("⚠️ 일부 고급 모델 라이브러리 없음 (XGBoost, LightGBM)")

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("🚀 고신뢰도 삼성전자 예측 시스템 로딩 완료!")


# 1. 최대 범위 데이터 수집기
class MaxDataCollector:
    def __init__(self):
        self.data_sources = {}

    def get_maximum_stock_data(self, symbol="005930.KS"):
        """최대한 많은 삼성전자 데이터 수집"""
        print("📊 최대 범위 삼성전자 데이터 수집 중...")

        # 다양한 기간으로 시도
        periods = ["max", "10y", "5y", "3y", "2y", "1y"]

        for period in periods:
            try:
                ticker = yf.Ticker(symbol)
                data = ticker.history(period=period)

                if not data.empty and len(data) > 500:  # 최소 500일 이상
                    # 타임존 제거
                    if hasattr(data.index, 'tz') and data.index.tz is not None:
                        data.index = data.index.tz_convert(None)

                    print(f"✅ 삼성전자 데이터: {len(data)}일 ({period}) - {data.index.min()} ~ {data.index.max()}")
                    return data

            except Exception as e:
                print(f"⚠️ {period} 수집 실패: {e}")
                continue

        raise Exception("충분한 주가 데이터를 가져올 수 없습니다.")

    def get_comprehensive_market_data(self, start_date=None, end_date=None):
        """포괄적인 시장 데이터 수집"""
        if start_date is None:
            start_date = datetime.now() - timedelta(days=3650)  # 10년
        if end_date is None:
            end_date = datetime.now()

        market_data = {}

        # 주요 글로벌 지수
        global_indices = {
            'KOSPI': '^KS11',
            'KOSDAQ': '^KQ11',
            'NASDAQ': '^IXIC',
            'SP500': '^GSPC',
            'DOW': '^DJI',
            'NIKKEI': '^N225',
            'FTSE': '^FTSE',
            'DAX': '^GDAXI',
            'HANG_SENG': '^HSI'
        }

        # 환율
        currencies = {
            'USD_KRW': 'KRW=X',
            'EUR_KRW': 'EURKRW=X',
            'JPY_KRW': 'JPYKRW=X',
            'CNY_KRW': 'CNYKRW=X',
            'DXY': 'DX-Y.NYB'  # 달러 인덱스
        }

        # 원자재
        commodities = {
            'GOLD': 'GC=F',
            'SILVER': 'SI=F',
            'OIL_WTI': 'CL=F',
            'OIL_BRENT': 'BZ=F',
            'COPPER': 'HG=F',
            'NATURAL_GAS': 'NG=F'
        }

        # 채권 수익률
        bonds = {
            'US_10Y': '^TNX',
            'US_2Y': '^IRX',
            'US_30Y': '^TYX',
            'KR_10Y': '000020.KS'  # 한국 10년 국채 (근사치)
        }

        # 변동성 지수
        volatility = {
            'VIX': '^VIX',
            'VIX9D': '^VIX9D',
            'VKOSPI': '^VKOSPI'
        }

        # 섹터 ETF
        sectors = {
            'TECH': 'XLK',
            'SEMICONDUCTOR': 'SMH',
            'SOXX': 'SOXX',
            'KOREA_ETF': 'EWY',
            'ASIA_TECH': 'AAXJ',
            'EMERGING_MARKETS': 'EEM'
        }

        all_symbols = {**global_indices, **currencies, **commodities,
                       **bonds, **volatility, **sectors}

        print(f"📈 {len(all_symbols)}개 시장 지표 수집 중...")

        collected_count = 0
        for name, symbol in all_symbols.items():
            try:
                ticker = yf.Ticker(symbol)
                data = ticker.history(start=start_date, end=end_date)

                if not data.empty:
                    # 타임존 제거
                    if hasattr(data.index, 'tz') and data.index.tz is not None:
                        data.index = data.index.tz_convert(None)

                    market_data[name] = data['Close']
                    collected_count += 1

                    if collected_count % 5 == 0:
                        print(f"📊 진행률: {collected_count}/{len(all_symbols)}")
                        time.sleep(0.5)  # API 제한 고려

            except Exception as e:
                print(f"⚠️ {name} 수집 실패: {e}")
                continue

        print(f"✅ 시장 데이터 수집 완료: {collected_count}개")
        return market_data

    def get_related_companies_data(self, start_date=None, end_date=None):
        """삼성전자 관련 회사들 데이터"""
        if start_date is None:
            start_date = datetime.now() - timedelta(days=1825)  # 5년

        # 삼성 계열사 및 경쟁사
        related_companies = {
            'SAMSUNG_SDI': '006400.KS',  # 삼성SDI
            'SAMSUNG_BIOLOGICS': '207940.KS',  # 삼성바이오로직스
            'SK_HYNIX': '000660.KS',  # SK하이닉스
            'LG_DISPLAY': '034220.KS',  # LG디스플레이
            'LG_ELEC': '066570.KS',  # LG전자
            # 글로벌 경쟁사
            'APPLE': 'AAPL',
            'TSMC': 'TSM',
            'NVIDIA': 'NVDA',
            'INTEL': 'INTC',
            'QUALCOMM': 'QCOM',
            'BROADCOM': 'AVGO',
            'SONY': 'SONY',
            'ASML': 'ASML'
        }

        company_data = {}

        for name, symbol in related_companies.items():
            try:
                ticker = yf.Ticker(symbol)
                data = ticker.history(start=start_date, end=end_date)

                if not data.empty:
                    if hasattr(data.index, 'tz') and data.index.tz is not None:
                        data.index = data.index.tz_convert(None)

                    company_data[f'related_{name}'] = data['Close']

            except Exception as e:
                print(f"⚠️ {name} 수집 실패: {e}")
                continue

        print(f"✅ 관련 회사 데이터: {len(company_data)}개")
        return company_data


# 2. 고급 기술적 지표 계산기
class AdvancedTechnicalIndicators:
    @staticmethod
    def calculate_all_indicators(data):
        """모든 고급 기술적 지표 계산"""
        df = data.copy()

        print("📈 고급 기술적 지표 계산 중...")

        # 기본 이동평균 (다양한 기간)
        ma_periods = [3, 5, 8, 10, 13, 20, 21, 34, 50, 89, 100, 144, 200]
        for period in ma_periods:
            if len(df) > period:
                df[f'MA_{period}'] = df['Close'].rolling(period).mean()
                df[f'EMA_{period}'] = df['Close'].ewm(span=period).mean()

        # 볼린저 밴드 (다양한 기간과 표준편차)
        for period in [10, 20, 50]:
            for std_dev in [1.5, 2.0, 2.5]:
                if len(df) > period:
                    ma = df['Close'].rolling(period).mean()
                    std = df['Close'].rolling(period).std()
                    df[f'BB_upper_{period}_{std_dev}'] = ma + (std * std_dev)
                    df[f'BB_lower_{period}_{std_dev}'] = ma - (std * std_dev)
                    df[f'BB_width_{period}_{std_dev}'] = df[f'BB_upper_{period}_{std_dev}'] - df[
                        f'BB_lower_{period}_{std_dev}']
                    df[f'BB_position_{period}_{std_dev}'] = (df['Close'] - df[f'BB_lower_{period}_{std_dev}']) / df[
                        f'BB_width_{period}_{std_dev}']

        # RSI (다양한 기간)
        for period in [9, 14, 21, 30]:
            if len(df) > period:
                delta = df['Close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
                rs = gain / loss
                df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

        # MACD (다양한 설정)
        macd_configs = [(12, 26, 9), (5, 35, 5), (8, 21, 5)]
        for fast, slow, signal in macd_configs:
            if len(df) > slow:
                ema_fast = df['Close'].ewm(span=fast).mean()
                ema_slow = df['Close'].ewm(span=slow).mean()
                macd = ema_fast - ema_slow
                macd_signal = macd.ewm(span=signal).mean()
                df[f'MACD_{fast}_{slow}'] = macd
                df[f'MACD_signal_{fast}_{slow}'] = macd_signal
                df[f'MACD_histogram_{fast}_{slow}'] = macd - macd_signal

        # 스토캐스틱
        for k_period, d_period in [(14, 3), (21, 5), (9, 3)]:
            if len(df) > k_period:
                low_k = df['Low'].rolling(k_period).min()
                high_k = df['High'].rolling(k_period).max()
                k_percent = 100 * ((df['Close'] - low_k) / (high_k - low_k))
                df[f'Stoch_K_{k_period}'] = k_percent
                df[f'Stoch_D_{k_period}'] = k_percent.rolling(d_period).mean()

        # Williams %R
        for period in [14, 21]:
            if len(df) > period:
                high_n = df['High'].rolling(period).max()
                low_n = df['Low'].rolling(period).min()
                df[f'Williams_R_{period}'] = -100 * (high_n - df['Close']) / (high_n - low_n)

        # ATR (Average True Range)
        for period in [14, 21, 30]:
            if len(df) > period:
                high_low = df['High'] - df['Low']
                high_close = np.abs(df['High'] - df['Close'].shift())
                low_close = np.abs(df['Low'] - df['Close'].shift())
                true_range = np.maximum(high_low, np.maximum(high_close, low_close))
                df[f'ATR_{period}'] = true_range.rolling(period).mean()

        # ADX (Average Directional Index)
        for period in [14, 21]:
            if len(df) > period:
                # 간단한 ADX 계산
                high_diff = df['High'].diff()
                low_diff = df['Low'].diff()
                pos_dm = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0)
                neg_dm = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0)

                tr = np.maximum(df['High'] - df['Low'],
                                np.maximum(abs(df['High'] - df['Close'].shift()),
                                           abs(df['Low'] - df['Close'].shift())))

                atr = pd.Series(tr).rolling(period).mean()
                pos_di = 100 * pd.Series(pos_dm).rolling(period).mean() / atr
                neg_di = 100 * pd.Series(neg_dm).rolling(period).mean() / atr

                df[f'ADX_{period}'] = 100 * abs(pos_di - neg_di) / (pos_di + neg_di)

        # 일목균형표
        if len(df) > 52:
            # 기준선 (Tenkan-sen)
            tenkan_high = df['High'].rolling(9).max()
            tenkan_low = df['Low'].rolling(9).min()
            df['Tenkan'] = (tenkan_high + tenkan_low) / 2

            # 전환선 (Kijun-sen)
            kijun_high = df['High'].rolling(26).max()
            kijun_low = df['Low'].rolling(26).min()
            df['Kijun'] = (kijun_high + kijun_low) / 2

            # 선행스팬 A
            df['Senkou_A'] = ((df['Tenkan'] + df['Kijun']) / 2).shift(26)

            # 선행스팬 B
            senkou_high = df['High'].rolling(52).max()
            senkou_low = df['Low'].rolling(52).min()
            df['Senkou_B'] = ((senkou_high + senkou_low) / 2).shift(26)

        # 가격 채널
        for period in [10, 20, 50]:
            if len(df) > period:
                df[f'Price_Channel_High_{period}'] = df['High'].rolling(period).max()
                df[f'Price_Channel_Low_{period}'] = df['Low'].rolling(period).min()
                df[f'Price_Channel_Mid_{period}'] = (df[f'Price_Channel_High_{period}'] + df[
                    f'Price_Channel_Low_{period}']) / 2
                df[f'Price_Channel_Position_{period}'] = (df['Close'] - df[f'Price_Channel_Low_{period}']) / (
                            df[f'Price_Channel_High_{period}'] - df[f'Price_Channel_Low_{period}'])

        # 모멘텀 지표
        for period in [1, 3, 5, 10, 20, 50]:
            if len(df) > period:
                df[f'Price_Momentum_{period}'] = df['Close'].pct_change(period)
                df[f'Volume_Momentum_{period}'] = df['Volume'].pct_change(period)

        # 변동성 지표
        for period in [5, 10, 20, 30, 50]:
            if len(df) > period:
                df[f'Price_Volatility_{period}'] = df['Close'].rolling(period).std()
                df[f'Returns_Volatility_{period}'] = df['Close'].pct_change().rolling(period).std()
                df[f'Volume_Volatility_{period}'] = df['Volume'].rolling(period).std()

        # 거래량 지표
        for period in [5, 10, 20, 50]:
            if len(df) > period:
                df[f'Volume_MA_{period}'] = df['Volume'].rolling(period).mean()
                df[f'Volume_Ratio_{period}'] = df['Volume'] / df[f'Volume_MA_{period}']

                # OBV (On Balance Volume)
                obv = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
                df[f'OBV_{period}'] = obv.rolling(period).mean()

        # 파라볼릭 SAR (간단한 버전)
        if len(df) > 20:
            af = 0.02
            max_af = 0.2
            sar = df['Close'].copy()
            trend = 1

            for i in range(1, len(df)):
                if trend == 1:  # 상승 추세
                    sar.iloc[i] = sar.iloc[i - 1] + af * (df['High'].iloc[i - 1] - sar.iloc[i - 1])
                    if df['Low'].iloc[i] < sar.iloc[i]:
                        trend = -1
                        sar.iloc[i] = df['High'].iloc[i - 1]
                        af = 0.02
                else:  # 하락 추세
                    sar.iloc[i] = sar.iloc[i - 1] + af * (df['Low'].iloc[i - 1] - sar.iloc[i - 1])
                    if df['High'].iloc[i] > sar.iloc[i]:
                        trend = 1
                        sar.iloc[i] = df['Low'].iloc[i - 1]
                        af = 0.02

            df['Parabolic_SAR'] = sar

        # 추가 복합 지표
        # 이동평균 교차 신호
        if 'MA_5' in df.columns and 'MA_20' in df.columns:
            df['MA_Cross_5_20'] = (df['MA_5'] > df['MA_20']).astype(int)
            df['MA_Cross_Signal_5_20'] = df['MA_Cross_5_20'].diff()

        if 'MA_20' in df.columns and 'MA_50' in df.columns:
            df['MA_Cross_20_50'] = (df['MA_20'] > df['MA_50']).astype(int)
            df['MA_Cross_Signal_20_50'] = df['MA_Cross_20_50'].diff()

        # RSI 기반 신호
        if 'RSI_14' in df.columns:
            df['RSI_Oversold'] = (df['RSI_14'] < 30).astype(int)
            df['RSI_Overbought'] = (df['RSI_14'] > 70).astype(int)
            df['RSI_Momentum'] = df['RSI_14'].diff()

        # 볼린저 밴드 기반 신호
        if 'BB_position_20_2.0' in df.columns:
            df['BB_Squeeze'] = (df['BB_width_20_2.0'] < df['BB_width_20_2.0'].rolling(20).quantile(0.2)).astype(int)
            df['BB_Breakout_Up'] = (df['Close'] > df['BB_upper_20_2.0']).astype(int)
            df['BB_Breakout_Down'] = (df['Close'] < df['BB_lower_20_2.0']).astype(int)

        print(f"✅ 고급 기술적 지표 계산 완료: {df.shape}")
        return df


# 3. 확장된 뉴스 감성 분석기
class AdvancedNewsAnalyzer:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def get_comprehensive_news(self, limit=100):
        """포괄적인 뉴스 수집"""
        all_news = []

        # 1. Yahoo Finance 뉴스 (더 많이)
        try:
            ticker = yf.Ticker("005930.KS")
            news = ticker.news

            for article in news[:limit // 2]:
                all_news.append({
                    'title': article.get('title', ''),
                    'summary': article.get('summary', ''),
                    'published': datetime.fromtimestamp(article.get('providerPublishTime', 0)).replace(tzinfo=None),
                    'source': 'Yahoo Finance',
                    'url': article.get('link', '')
                })
        except Exception as e:
            print(f"⚠️ Yahoo 뉴스 수집 실패: {e}")

        # 2. Google News RSS (여러 키워드)
        keywords = ['삼성전자', 'Samsung Electronics', 'Samsung', '005930',
                    'Samsung chip', 'Samsung semiconductor', '삼성 반도체']

        for keyword in keywords:
            try:
                url = f"https://news.google.com/rss/search?q={keyword}&hl=ko&gl=KR&ceid=KR:ko"
                feed = feedparser.parse(url)

                for entry in feed.entries[:limit // len(keywords)]:
                    pub_time = entry.published_parsed
                    if pub_time:
                        published = datetime(*pub_time[:6])
                    else:
                        published = datetime.now() - timedelta(days=1)

                    all_news.append({
                        'title': entry.title,
                        'summary': getattr(entry, 'summary', ''),
                        'published': published,
                        'source': f'Google News ({keyword})',
                        'url': entry.link
                    })

                time.sleep(0.5)  # API 제한 고려

            except Exception as e:
                print(f"⚠️ Google 뉴스 ({keyword}) 실패: {e}")
                continue

        if all_news:
            news_df = pd.DataFrame(all_news)

            # 중복 제거 (제목 기준)
            news_df = news_df.drop_duplicates(subset=['title'], keep='first')

            # 최근 순으로 정렬
            news_df = news_df.sort_values('published', ascending=False)

            print(f"✅ 뉴스 수집 완료: {len(news_df)}개")
            return news_df

        return pd.DataFrame()

    def advanced_sentiment_analysis(self, news_df):
        """고급 감성 분석"""
        if news_df.empty:
            return pd.DataFrame()

        print("🧠 고급 감성 분석 수행 중...")

        enhanced_news = news_df.copy()

        # 다중 감성 분석
        sentiment_results = []

        for idx, row in enhanced_news.iterrows():
            text = f"{row['title']} {row['summary']}"

            # VADER 감성 분석
            vader_scores = self.analyzer.polarity_scores(text)

            # TextBlob 감성 분석
            blob = TextBlob(text)

            # 키워드 기반 감성 가중치
            positive_keywords = ['성장', '증가', '상승', '호조', '개선', '확대', '혁신', '성공',
                                 'growth', 'increase', 'rise', 'improve', 'success', 'innovation']
            negative_keywords = ['하락', '감소', '우려', '위험', '손실', '어려움', '하향', '부진',
                                 'decline', 'decrease', 'concern', 'risk', 'loss', 'difficulty']

            positive_weight = sum(1 for word in positive_keywords if word in text.lower())
            negative_weight = sum(1 for word in negative_keywords if word in text.lower())

            # 종합 감성 점수 계산
            base_sentiment = (vader_scores['compound'] + blob.sentiment.polarity) / 2
            keyword_adjustment = (positive_weight - negative_weight) * 0.1
            final_sentiment = np.clip(base_sentiment + keyword_adjustment, -1, 1)

            sentiment_results.append({
                'compound': final_sentiment,
                'vader_compound': vader_scores['compound'],
                'vader_positive': vader_scores['pos'],
                'vader_negative': vader_scores['neg'],
                'vader_neutral': vader_scores['neu'],
                'textblob_polarity': blob.sentiment.polarity,
                'textblob_subjectivity': blob.sentiment.subjectivity,
                'positive_keywords': positive_weight,
                'negative_keywords': negative_weight,
                'news_importance': len(text) / 100  # 뉴스 길이 기반 중요도
            })

        sentiment_df = pd.DataFrame(sentiment_results)
        enhanced_news = pd.concat([enhanced_news.reset_index(drop=True), sentiment_df], axis=1)

        # 시간별 감성 집계 (더 세밀하게)
        enhanced_news['date'] = pd.to_datetime(enhanced_news['published']).dt.date
        enhanced_news['hour'] = pd.to_datetime(enhanced_news['published']).dt.hour

        # 일별 감성 집계
        daily_sentiment = enhanced_news.groupby('date').agg({
            'compound': ['mean', 'std', 'count', 'min', 'max'],
            'vader_positive': 'mean',
            'vader_negative': 'mean',
            'textblob_polarity': ['mean', 'std'],
            'news_importance': 'sum',
            'positive_keywords': 'sum',
            'negative_keywords': 'sum'
        }).round(4)

        daily_sentiment.columns = ['_'.join(col).strip() for col in daily_sentiment.columns]
        daily_sentiment = daily_sentiment.reset_index()
        daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

        # 감성 모멘텀 계산
        daily_sentiment['sentiment_momentum_3d'] = daily_sentiment['compound_mean'].rolling(3).mean()
        daily_sentiment['sentiment_momentum_7d'] = daily_sentiment['compound_mean'].rolling(7).mean()
        daily_sentiment['sentiment_volatility_7d'] = daily_sentiment['compound_mean'].rolling(7).std()

        # 뉴스 활동성 지표
        daily_sentiment['news_activity_score'] = (
                daily_sentiment['compound_count'] * 0.3 +
                daily_sentiment['news_importance_sum'] * 0.4 +
                daily_sentiment['compound_std'].fillna(0) * 0.3
        )

        print(f"✅ 고급 감성 분석 완료: {len(daily_sentiment)}일")
        return daily_sentiment


# 4. 데이터 통합 및 고급 피처 엔지니어링
class ComprehensiveFeatureEngineering:
    def __init__(self):
        pass

    def integrate_all_data(self, stock_data, market_data, company_data, news_sentiment):
        """모든 데이터 통합"""
        print("🔧 데이터 통합 중...")

        # 주가 데이터를 기준으로 시작
        integrated_df = stock_data.copy()

        # 시장 데이터 통합
        for name, series in market_data.items():
            try:
                # 일별 리샘플링
                series_daily = series.resample('D').last().fillna(method='ffill')
                integrated_df = integrated_df.merge(
                    series_daily.rename(f'market_{name}'),
                    left_index=True, right_index=True, how='left'
                )
            except Exception as e:
                print(f"⚠️ {name} 통합 실패: {e}")
                continue

        # 관련 회사 데이터 통합
        for name, series in company_data.items():
            try:
                series_daily = series.resample('D').last().fillna(method='ffill')
                integrated_df = integrated_df.merge(
                    series_daily.rename(name),
                    left_index=True, right_index=True, how='left'
                )
            except Exception as e:
                print(f"⚠️ {name} 통합 실패: {e}")
                continue

        # 뉴스 감성 데이터 통합
        if not news_sentiment.empty:
            try:
                news_sentiment.set_index('date', inplace=True)
                integrated_df = integrated_df.merge(
                    news_sentiment,
                    left_index=True, right_index=True, how='left'
                )

                # 감성 데이터 전방향 채우기
                sentiment_cols = [col for col in news_sentiment.columns if 'compound' in col or 'sentiment' in col]
                for col in sentiment_cols:
                    if col in integrated_df.columns:
                        integrated_df[col] = integrated_df[col].fillna(method='ffill').fillna(0)

            except Exception as e:
                print(f"⚠️ 뉴스 데이터 통합 실패: {e}")

        print(f"✅ 데이터 통합 완료: {integrated_df.shape}")
        return integrated_df

    def create_advanced_features(self, df):
        """고급 피처 생성"""
        print("🧬 고급 피처 엔지니어링 중...")

        feature_df = df.copy()

        # 1. 시장 상관관계 피처
        if 'market_KOSPI' in feature_df.columns:
            # 삼성전자와 KOSPI 상관관계
            for window in [5, 10, 20, 50]:
                feature_df[f'samsung_kospi_corr_{window}d'] = feature_df['Close'].rolling(window).corr(
                    feature_df['market_KOSPI'])
                feature_df[f'samsung_kospi_ratio_{window}d'] = feature_df['Close'] / feature_df['market_KOSPI']
                feature_df[f'samsung_kospi_spread_{window}d'] = feature_df['Close'].pct_change(window) - feature_df[
                    'market_KOSPI'].pct_change(window)

        # 2. 글로벌 시장 영향 피처
        global_markets = ['market_NASDAQ', 'market_SP500', 'market_NIKKEI']
        available_markets = [m for m in global_markets if m in feature_df.columns]

        if available_markets:
            # 글로벌 시장 평균
            global_avg = feature_df[available_markets].mean(axis=1)
            feature_df['global_market_avg'] = global_avg
            feature_df['samsung_global_corr'] = feature_df['Close'].rolling(20).corr(global_avg)

        # 3. 환율 영향 피처
        if 'market_USD_KRW' in feature_df.columns:
            for window in [1, 5, 10, 20]:
                feature_df[f'usd_krw_change_{window}d'] = feature_df['market_USD_KRW'].pct_change(window)
                feature_df[f'samsung_fx_impact_{window}d'] = feature_df['Close'].pct_change(window) * feature_df[
                    f'usd_krw_change_{window}d']

        # 4. 변동성 클러스터 피처
        returns = feature_df['Close'].pct_change()
        for window in [5, 10, 20]:
            volatility = returns.rolling(window).std()
            feature_df[f'volatility_regime_{window}d'] = pd.qcut(volatility.rank(method='first'), 5,
                                                                 labels=[0, 1, 2, 3, 4])
            feature_df[f'volatility_breakout_{window}d'] = (
                        volatility > volatility.rolling(window * 2).quantile(0.8)).astype(int)

        # 5. 거래량 프로파일 피처
        for window in [5, 10, 20]:
            volume_ma = feature_df['Volume'].rolling(window).mean()
            feature_df[f'volume_profile_{window}d'] = feature_df['Volume'] / volume_ma
            feature_df[f'volume_breakout_{window}d'] = (feature_df['Volume'] > volume_ma * 2).astype(int)

            # 가격-거래량 관계
            price_volume_corr = feature_df['Close'].rolling(window).corr(feature_df['Volume'])
            feature_df[f'price_volume_corr_{window}d'] = price_volume_corr

        # 6. 기술적 신호 조합 피처
        # 다중 RSI 신호
        rsi_cols = [col for col in feature_df.columns if col.startswith('RSI_')]
        if len(rsi_cols) >= 2:
            feature_df['rsi_consensus'] = feature_df[rsi_cols].mean(axis=1)
            feature_df['rsi_divergence'] = feature_df[rsi_cols].std(axis=1)

        # 다중 이동평균 신호
        ma_cols = [col for col in feature_df.columns if col.startswith('MA_')]
        if len(ma_cols) >= 3:
            # 현재 가격이 이동평균들 위에 있는 비율
            ma_above = (feature_df['Close'].values[:, np.newaxis] > feature_df[ma_cols].values).mean(axis=1)
            feature_df['ma_consensus'] = ma_above

        # 7. 시장 스트레스 지수
        stress_indicators = []
        if 'market_VIX' in feature_df.columns:
            stress_indicators.append((feature_df['market_VIX'] > 25).astype(int))
        if 'compound_mean' in feature_df.columns:
            stress_indicators.append((feature_df['compound_mean'] < -0.3).astype(int))
        if 'volatility_regime_20d' in feature_df.columns:
            stress_indicators.append((feature_df['volatility_regime_20d'] >= 3).astype(int))

        if stress_indicators:
            feature_df['market_stress_index'] = np.mean(stress_indicators, axis=0)

        # 8. 계절성 피처
        feature_df['day_of_week'] = feature_df.index.dayofweek
        feature_df['month'] = feature_df.index.month
        feature_df['quarter'] = feature_df.index.quarter
        feature_df['is_month_end'] = (feature_df.index == feature_df.index.to_period('M').end_time).astype(int)
        feature_df['is_quarter_end'] = (feature_df.index == feature_df.index.to_period('Q').end_time).astype(int)

        # 9. 경쟁사 관계 피처
        competitor_cols = [col for col in feature_df.columns if
                           'related_' in col and 'HYNIX' in col or 'APPLE' in col or 'TSMC' in col]
        for comp_col in competitor_cols:
            if comp_col in feature_df.columns:
                # 상대적 성과
                for window in [5, 20]:
                    samsung_ret = feature_df['Close'].pct_change(window)
                    comp_ret = feature_df[comp_col].pct_change(window)
                    feature_df[f'relative_performance_{comp_col}_{window}d'] = samsung_ret - comp_ret

        # 10. 라그 피처 (시차 변수)
        important_features = ['Close', 'Volume', 'market_KOSPI', 'market_VIX']
        available_features = [f for f in important_features if f in feature_df.columns]

        for feature in available_features:
            for lag in [1, 2, 3, 5]:
                feature_df[f'{feature}_lag_{lag}d'] = feature_df[feature].shift(lag)

        # 11. 모멘텀 피처 (다양한 기간)
        for window in [3, 5, 10, 20, 50]:
            if len(feature_df) > window:
                # 가격 모멘텀
                feature_df[f'price_momentum_{window}d'] = feature_df['Close'] / feature_df['Close'].shift(window) - 1

                # 가속도 (모멘텀의 변화율)
                if window <= 20:
                    momentum = feature_df['Close'].pct_change(window)
                    feature_df[f'price_acceleration_{window}d'] = momentum.diff()

        # 결측치 처리
        print("🔧 결측치 처리 중...")

        # 숫자형 컬럼만 선택
        numeric_cols = feature_df.select_dtypes(include=[np.number]).columns

        # 전방향 채우기 -> 후방향 채우기 -> 0으로 채우기
        feature_df[numeric_cols] = feature_df[numeric_cols].fillna(method='ffill').fillna(method='bfill').fillna(0)

        print(f"✅ 고급 피처 엔지니어링 완료: {feature_df.shape}")
        return feature_df


# 5. 고신뢰도 예측 모델 시스템
class HighReliabilityPredictionSystem:
    def __init__(self):
        self.models = {}
        self.feature_importance = {}
        self.scalers = {}
        self.feature_selector = None
        self.ensemble_weights = {}

    def prepare_training_data(self, data, target_days=5, validation_split=0.2):
        """고급 훈련 데이터 준비"""
        print("📊 고급 훈련 데이터 준비 중...")

        # 타겟 변수 생성 (다양한 예측 기간)
        target_periods = [1, 3, 5, 10]
        for days in target_periods:
            data[f'target_return_{days}d'] = data['Close'].pct_change(days).shift(-days)
            data[f'target_direction_{days}d'] = (data[f'target_return_{days}d'] > 0).astype(int)
            data[f'target_volatility_{days}d'] = data['Close'].pct_change().rolling(days).std().shift(-days)

        # 기본 타겟 설정
        main_target = f'target_return_{target_days}d'

        # 피처 선택 (더 엄격한 기준)
        exclude_patterns = ['target_', 'Adj Close', 'Open', 'High', 'Low']
        feature_cols = []

        for col in data.columns:
            if (data[col].dtype in ['float64', 'int64'] and
                    not any(pattern in col for pattern in exclude_patterns) and
                    data[col].count() >= len(data) * 0.7):  # 70% 이상 데이터 있어야 함
                feature_cols.append(col)

        print(f"📋 후보 피처: {len(feature_cols)}개")

        # 결측치 제거
        required_cols = feature_cols + [main_target]
        clean_data = data[required_cols].dropna()

        if len(clean_data) < 100:
            raise Exception(f"정제된 데이터가 부족합니다 (현재: {len(clean_data)}개, 최소: 100개 필요)")

        print(f"✅ 정제된 데이터: {clean_data.shape}")

        # 시계열 분할 (더 정교하게)
        train_size = int(len(clean_data) * (1 - validation_split))

        X_train = clean_data[feature_cols].iloc[:train_size]
        X_val = clean_data[feature_cols].iloc[train_size:]
        y_train = clean_data[main_target].iloc[:train_size]
        y_val = clean_data[main_target].iloc[train_size:]

        return X_train, X_val, y_train, y_val, feature_cols

    def feature_selection(self, X_train, y_train, feature_cols, max_features=50):
        """고급 피처 선택"""
        print("🔍 피처 선택 중...")

        # 1. 기본 통계적 필터링
        selector_stats = SelectKBest(score_func=f_regression, k=min(max_features * 2, len(feature_cols)))
        X_selected = selector_stats.fit_transform(X_train, y_train)
        selected_features_idx = selector_stats.get_support(indices=True)
        selected_features = [feature_cols[i] for i in selected_features_idx]

        # 2. 재귀적 피처 제거 (RFE)
        rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
        rfe_selector = RFE(rf_model, n_features_to_select=min(max_features, len(selected_features)))
        rfe_selector.fit(X_train[selected_features], y_train)

        final_features = [selected_features[i] for i in range(len(selected_features)) if rfe_selector.support_[i]]

        print(f"✅ 최종 선택된 피처: {len(final_features)}개")
        return final_features

    def train_advanced_models(self, X_train, X_val, y_train, y_val, selected_features):
        """고급 모델들 훈련"""
        print("🔥 고급 모델 훈련 시작...")

        # 선택된 피처만 사용
        X_train_selected = X_train[selected_features]
        X_val_selected = X_val[selected_features]

        # 다양한 스케일러 준비
        scalers = {
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }

        # 확장된 모델 설정
        model_configs = {
            'RandomForest': {
                'model': RandomForestRegressor(
                    n_estimators=200,
                    max_depth=min(15, len(selected_features)),
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=-1
                ),
                'scaler': None
            },
            'ExtraTrees': {
                'model': ExtraTreesRegressor(
                    n_estimators=200,
                    max_depth=min(15, len(selected_features)),
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=-1
                ),
                'scaler': None
            },
            'GradientBoosting': {
                'model': GradientBoostingRegressor(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.1,
                    subsample=0.8,
                    random_state=42
                ),
                'scaler': None
            },
            'XGBoost': {
                'model': xgb.XGBRegressor(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.1,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    n_jobs=-1
                ) if 'xgb' in globals() else None,
                'scaler': None
            },
            'LightGBM': {
                'model': lgb.LGBMRegressor(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.1,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    verbose=-1
                ) if 'lgb' in globals() else None,
                'scaler': None
            },
            'Ridge': {
                'model': Ridge(alpha=1.0),
                'scaler': 'standard'
            },
            'ElasticNet': {
                'model': ElasticNet(alpha=0.1, l1_ratio=0.5),
                'scaler': 'standard'
            },
            'SVR': {
                'model': SVR(kernel='rbf', C=1.0, epsilon=0.01),
                'scaler': 'robust'
            },
            'NeuralNetwork': {
                'model': MLPRegressor(
                    hidden_layer_sizes=(100, 50, 25),
                    max_iter=500,
                    learning_rate_init=0.001,
                    random_state=42
                ),
                'scaler': 'standard'
            }
        }

        results = {}
        valid_models = 0

        for model_name, config in model_configs.items():
            if config['model'] is None:
                continue

            print(f"🔥 {model_name} 훈련 중...")

            try:
                model = config['model']
                scaler_name = config['scaler']

                # 데이터 준비
                if scaler_name:
                    scaler = scalers[scaler_name]
                    X_train_final = scaler.fit_transform(X_train_selected)
                    X_val_final = scaler.transform(X_val_selected)
                    self.scalers[model_name] = scaler
                else:
                    X_train_final = X_train_selected.values
                    X_val_final = X_val_selected.values
                    self.scalers[model_name] = None

                # 모델 훈련
                model.fit(X_train_final, y_train)

                # 예측
                y_train_pred = model.predict(X_train_final)
                y_val_pred = model.predict(X_val_final)

                # 성능 평가
                train_r2 = r2_score(y_train, y_train_pred)
                val_r2 = r2_score(y_val, y_val_pred)
                val_mse = mean_squared_error(y_val, y_val_pred)
                val_mae = mean_absolute_error(y_val, y_val_pred)

                # 과적합 체크
                overfitting = train_r2 - val_r2

                # 교차 검증 (시계열용)
                tscv = TimeSeriesSplit(n_splits=5)
                cv_scores = cross_val_score(model, X_train_final, y_train, cv=tscv, scoring='r2')
                cv_mean = cv_scores.mean()
                cv_std = cv_scores.std()

                results[model_name] = {
                    'model': model,
                    'train_r2': train_r2,
                    'val_r2': val_r2,
                    'val_mse': val_mse,
                    'val_mae': val_mae,
                    'overfitting': overfitting,
                    'cv_mean': cv_mean,
                    'cv_std': cv_std,
                    'predictions': y_val_pred,
                    'stability_score': cv_mean - cv_std - abs(overfitting)  # 안정성 점수
                }

                # 피처 중요도 저장
                if hasattr(model, 'feature_importances_'):
                    importance_df = pd.DataFrame({
                        'feature': selected_features,
                        'importance': model.feature_importances_
                    }).sort_values('importance', ascending=False)
                    self.feature_importance[model_name] = importance_df

                valid_models += 1
                print(f"✅ {model_name} - 검증 R²: {val_r2:.4f}, CV: {cv_mean:.4f}±{cv_std:.4f}")

            except Exception as e:
                print(f"❌ {model_name} 훈련 실패: {e}")
                continue

        if valid_models == 0:
            raise Exception("모든 모델 훈련에 실패했습니다.")

        # 모델 필터링 (성능이 너무 낮은 모델 제거)
        filtered_results = {}
        for name, result in results.items():
            if result['val_r2'] > -0.5 and result['stability_score'] > -1.0:  # 기본적인 성능 기준
                filtered_results[name] = result

        if not filtered_results:
            print("⚠️ 성능 기준을 만족하는 모델이 없어 모든 모델을 사용합니다.")
            filtered_results = results

        self.models = filtered_results
        print(f"✅ 총 {len(filtered_results)}개 모델 훈련 완료")

        return filtered_results

    def create_meta_ensemble(self, X_val, y_val, selected_features):
        """메타 앙상블 생성"""
        print("🎭 메타 앙상블 구성 중...")

        # 각 모델의 예측값을 새로운 피처로 사용
        meta_features = []

        for model_name, model_info in self.models.items():
            predictions = model_info['predictions']
            meta_features.append(predictions)

        if len(meta_features) < 2:
            print("⚠️ 메타 앙상블을 위한 모델이 부족합니다.")
            return None

        # 메타 피처 행렬 생성
        meta_X = np.column_stack(meta_features)

        # 메타 모델 (간단한 선형 모델 사용)
        meta_model = Ridge(alpha=0.1)
        meta_model.fit(meta_X, y_val)

        # 메타 모델 성능
        meta_pred = meta_model.predict(meta_X)
        meta_r2 = r2_score(y_val, meta_pred)

        print(f"✅ 메타 앙상블 R²: {meta_r2:.4f}")

        return meta_model

    def calculate_ensemble_weights(self):
        """동적 앙상블 가중치 계산"""
        weights = {}

        # 다양한 가중치 방법
        for model_name, result in self.models.items():
            # 1. 성능 기반 가중치
            performance_weight = max(0, result['val_r2'])

            # 2. 안정성 기반 가중치
            stability_weight = max(0, result['stability_score'])

            # 3. 과적합 페널티
            overfitting_penalty = max(0, 1 - abs(result['overfitting']))

            # 종합 가중치
            final_weight = (performance_weight * 0.4 +
                            stability_weight * 0.4 +
                            overfitting_penalty * 0.2)

            weights[model_name] = final_weight

        # 정규화
        total_weight = sum(weights.values())
        if total_weight > 0:
            weights = {k: v / total_weight for k, v in weights.items()}

        self.ensemble_weights = weights
        return weights

    def make_ensemble_prediction(self, latest_data, selected_features, confidence_level=0.95):
        """신뢰구간을 포함한 앙상블 예측"""
        if not self.models:
            raise Exception("훈련된 모델이 없습니다.")

        # 최신 데이터 준비
        latest_features = latest_data[selected_features].iloc[-1:].fillna(0)

        predictions = []
        model_weights = []

        for model_name, model_info in self.models.items():
            try:
                model = model_info['model']
                scaler = self.scalers.get(model_name)

                # 데이터 전처리
                if scaler:
                    X_scaled = scaler.transform(latest_features)
                    pred = model.predict(X_scaled)[0]
                else:
                    pred = model.predict(latest_features.values)[0]

                predictions.append(pred)
                model_weights.append(self.ensemble_weights.get(model_name, 0))

            except Exception as e:
                print(f"⚠️ {model_name} 예측 실패: {e}")
                continue

        if not predictions:
            raise Exception("예측 생성에 실패했습니다.")

        # 가중 평균 예측
        predictions = np.array(predictions)
        weights = np.array(model_weights)

        if weights.sum() > 0:
            weights = weights / weights.sum()
            ensemble_pred = np.average(predictions, weights=weights)
        else:
            ensemble_pred = np.mean(predictions)

        # 예측 신뢰도 계산
        pred_std = np.std(predictions)
        confidence_interval = 1.96 * pred_std  # 95% 신뢰구간

        # 신뢰도 점수 (0-100)
        confidence_score = max(0, min(100, 100 * (1 - pred_std / max(abs(ensemble_pred), 0.01))))

        return {
            'prediction': ensemble_pred,
            'confidence_score': confidence_score,
            'confidence_interval': confidence_interval,
            'individual_predictions': dict(zip(self.models.keys(), predictions)),
            'prediction_std': pred_std,
            'model_weights': dict(zip(self.models.keys(), weights)) if weights.sum() > 0 else {}
        }


# 6. 메인 시스템 실행
def run_high_reliability_system():
    """고신뢰도 삼성전자 예측 시스템 실행"""
    print("🚀 고신뢰도 삼성전자 예측 시스템 시작!")
    print("=" * 80)

    try:
        # 1. 최대 데이터 수집
        print("📊 1단계: 최대 범위 데이터 수집")
        collector = MaxDataCollector()

        stock_data = collector.get_maximum_stock_data()
        start_date = stock_data.index.min()
        end_date = stock_data.index.max()

        market_data = collector.get_comprehensive_market_data(start_date, end_date)
        company_data = collector.get_related_companies_data(start_date, end_date)

        print(f"📈 수집된 데이터:")
        print(f"  - 주가 데이터: {len(stock_data)}일")
        print(f"  - 시장 지표: {len(market_data)}개")
        print(f"  - 관련 회사: {len(company_data)}개")

        # 2. 고급 기술적 지표 계산
        print("\n📈 2단계: 고급 기술적 지표 계산")
        technical_data = AdvancedTechnicalIndicators.calculate_all_indicators(stock_data)

        # 3. 뉴스 감성 분석
        print("\n📰 3단계: 뉴스 감성 분석")
        news_analyzer = AdvancedNewsAnalyzer()
        news_data = news_analyzer.get_comprehensive_news()
        news_sentiment = news_analyzer.advanced_sentiment_analysis(news_data)

        # 4. 데이터 통합 및 고급 피처 엔지니어링
        print("\n🔧 4단계: 데이터 통합 및 피처 엔지니어링")
        feature_engineer = ComprehensiveFeatureEngineering()
        integrated_data = feature_engineer.integrate_all_data(
            technical_data, market_data, company_data, news_sentiment
        )
        final_data = feature_engineer.create_advanced_features(integrated_data)

        print(f"📊 최종 통합 데이터: {final_data.shape}")
        print(f"📅 데이터 기간: {final_data.index.min()} ~ {final_data.index.max()}")

        # 5. 고신뢰도 모델 훈련
        print("\n🧠 5단계: 고신뢰도 모델 훈련")
        prediction_system = HighReliabilityPredictionSystem()

        # 훈련 데이터 준비
        X_train, X_val, y_train, y_val, feature_cols = prediction_system.prepare_training_data(final_data)

        # 피처 선택
        selected_features = prediction_system.feature_selection(X_train, y_train, feature_cols)

        # 모델 훈련
        model_results = prediction_system.train_advanced_models(
            X_train, X_val, y_train, y_val, selected_features
        )

        # 앙상블 가중치 계산
        ensemble_weights = prediction_system.calculate_ensemble_weights()

        # 6. 예측 수행
        print("\n🔮 6단계: 예측 수행")
        prediction_result = prediction_system.make_ensemble_prediction(
            final_data, selected_features
        )

        # 7. 결과 분석 및 출력
        current_price = final_data['Close'].iloc[-1]
        predicted_return = prediction_result['prediction']
        predicted_price = current_price * (1 + predicted_return)
        confidence_score = prediction_result['confidence_score']

        # 결과 출력
        print("\n" + "🎯 최종 예측 결과 " + "=" * 60)
        print(f"📊 현재 주가: {current_price:,.0f}원")
        print(f"🔮 5일 후 예상 수익률: {predicted_return * 100:+.2f}%")
        print(f"💰 5일 후 예상 주가: {predicted_price:,.0f}원")
        print(f"🎯 예측 신뢰도: {confidence_score:.1f}/100")
        print(f"📈 훈련 데이터 수: {len(X_train)}개")
        print(f"🧬 사용된 피처 수: {len(selected_features)}개")
        print(f"🤖 앙상블 모델 수: {len(model_results)}개")

        # 신뢰구간
        ci = prediction_result['confidence_interval']
        lower_bound = (predicted_return - ci) * 100
        upper_bound = (predicted_return + ci) * 100
        print(f"📊 95% 신뢰구간: {lower_bound:+.2f}% ~ {upper_bound:+.2f}%")

        # 개별 모델 예측
        print(f"\n📋 개별 모델 예측:")
        for model, pred in prediction_result['individual_predictions'].items():
            weight = ensemble_weights.get(model, 0)
            print(f"  {model}: {pred * 100:+.2f}% (가중치: {weight:.3f})")

        # 모델 성능
        print(f"\n📊 모델 성능 요약:")
        for model, result in model_results.items():
            print(f"  {model}: 검증 R² = {result['val_r2']:.4f}, 안정성 = {result['stability_score']:.4f}")

        # 8. 상세 분석 시각화
        create_comprehensive_dashboard(final_data, model_results, prediction_result, current_price, predicted_price)

        # 9. 결과 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # 데이터 저장
        final_data.to_csv(f'samsung_high_reliability_{timestamp}.csv')

        # 예측 결과 저장
        result_summary = {
            'timestamp': timestamp,
            'current_price': current_price,
            'predicted_return': predicted_return,
            'predicted_price': predicted_price,
            'confidence_score': confidence_score,
            'confidence_interval': ci,
            'model_count': len(model_results),
            'feature_count': len(selected_features),
            'training_samples': len(X_train),
            'data_period_days': (final_data.index.max() - final_data.index.min()).days,
            'individual_predictions': prediction_result['individual_predictions'],
            'ensemble_weights': ensemble_weights
        }

        with open(f'prediction_result_{timestamp}.json', 'w') as f:
            json.dump(result_summary, f, indent=2, default=str)

        print(f"\n✅ 결과 저장 완료:")
        print(f"  - 데이터: samsung_high_reliability_{timestamp}.csv")
        print(f"  - 예측 결과: prediction_result_{timestamp}.json")

        return final_data, model_results, prediction_result

    except Exception as e:
        print(f"❌ 시스템 실행 오류: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None


def create_comprehensive_dashboard(data, model_results, prediction_result, current_price, predicted_price):
    """종합 대시보드 생성"""
    try:
        print("📊 종합 대시보드 생성 중...")

        fig, axes = plt.subplots(3, 3, figsize=(20, 15))
        fig.suptitle('삼성전자 고신뢰도 예측 시스템 분석 대시보드', fontsize=16, fontweight='bold')

        # 1. 주가 차트 (최근 100일)
        recent_data = data.tail(100)
        axes[0, 0].plot(recent_data.index, recent_data['Close'], 'b-', linewidth=2, label='주가')
        if 'MA_20' in recent_data.columns:
            axes[0, 0].plot(recent_data.index, recent_data['MA_20'], 'r--', alpha=0.7, label='MA20')
        if 'MA_50' in recent_data.columns:
            axes[0, 0].plot(recent_data.index, recent_data['MA_50'], 'g--', alpha=0.7, label='MA50')
        axes[0, 0].set_title('주가 추이 (최근 100일)')
        axes[0, 0].set_ylabel('주가 (원)')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # 2. 거래량
        axes[0, 1].bar(recent_data.index, recent_data['Volume'], alpha=0.7, color='orange')
        axes[0, 1].set_title('거래량')
        axes[0, 1].set_ylabel('거래량')

        # 3. 모델 성능 비교
        if model_results:
            models = list(model_results.keys())
            r2_scores = [model_results[m]['val_r2'] for m in models]
            colors = ['green' if r2 > 0.3 else 'orange' if r2 > 0 else 'red' for r2 in r2_scores]
            bars = axes[0, 2].bar(models, r2_scores, color=colors)
            axes[0, 2].set_title('모델 성능 (R² Score)')
            axes[0, 2].set_ylabel('R² Score')
            axes[0, 2].tick_params(axis='x', rotation=45)

            # 값 표시
            for bar, score in zip(bars, r2_scores):
                height = bar.get_height()
                axes[0, 2].text(bar.get_x() + bar.get_width() / 2., height + 0.01,
                                f'{score:.3f}', ha='center', va='bottom')

        # 4. 예측 결과
        pred_data = ['현재', '5일 후 예상']
        pred_values = [current_price, predicted_price]
        colors = ['blue', 'green' if predicted_price > current_price else 'red']
        bars = axes[1, 0].bar(pred_data, pred_values, color=colors)
        axes[1, 0].set_title('주가 예측')
        axes[1, 0].set_ylabel('주가 (원)')

        # 값 표시
        for bar, value in zip(bars, pred_values):
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width() / 2., height + height * 0.01,
                            f'{value:,.0f}원', ha='center', va='bottom')

        # 5. 개별 모델 예측 분포
        if 'individual_predictions' in prediction_result:
            individual_preds = list(prediction_result['individual_predictions'].values())
            individual_returns = np.array(individual_preds) * 100

            axes[1, 1].hist(individual_returns, bins=min(10, len(individual_returns)),
                            alpha=0.7, color='skyblue', edgecolor='black')
            axes[1, 1].axvline(prediction_result['prediction'] * 100, color='red',
                               linestyle='--', linewidth=2, label='앙상블 예측')
            axes[1, 1].set_title('개별 모델 예측 분포')
            axes[1, 1].set_xlabel('예상 수익률 (%)')
            axes[1, 1].set_ylabel('빈도')
            axes[1, 1].legend()

        # 6. 신뢰도 지표
        confidence_score = prediction_result.get('confidence_score', 0)
        confidence_color = 'green' if confidence_score > 70 else 'orange' if confidence_score > 40 else 'red'

        # 파이 차트로 신뢰도 표시
        sizes = [confidence_score, 100 - confidence_score]
        colors = [confidence_color, 'lightgray']
        axes[1, 2].pie(sizes, labels=['신뢰도', '불확실성'], colors=colors, autopct='%1.1f%%',
                       startangle=90, counterclock=False)
        axes[1, 2].set_title(f'예측 신뢰도: {confidence_score:.1f}/100')

        # 7. 기술적 지표 (RSI)
        if 'RSI_14' in recent_data.columns:
            axes[2, 0].plot(recent_data.index, recent_data['RSI_14'], 'purple', linewidth=2)
            axes[2, 0].axhline(y=70, color='r', linestyle='--', alpha=0.7, label='과매수 (70)')
            axes[2, 0].axhline(y=30, color='g', linestyle='--', alpha=0.7, label='과매도 (30)')
            axes[2, 0].set_title('RSI (14일)')
            axes[2, 0].set_ylabel('RSI')
            axes[2, 0].legend()
            axes[2, 0].grid(True, alpha=0.3)

        # 8. 변동성
        if 'Price_Volatility_20' in recent_data.columns:
            axes[2, 1].plot(recent_data.index, recent_data['Price_Volatility_20'], 'brown', linewidth=2)
            axes[2, 1].set_title('가격 변동성 (20일)')
            axes[2, 1].set_ylabel('변동성')
            axes[2, 1].grid(True, alpha=0.3)

        # 9. 모델 가중치
        if 'model_weights' in prediction_result and prediction_result['model_weights']:
            weights = prediction_result['model_weights']
            models = list(weights.keys())
            weight_values = list(weights.values())

            axes[2, 2].pie(weight_values, labels=models, autopct='%1.2f')
            axes[2, 2].set_title('앙상블 모델 가중치')

        plt.tight_layout()
        plt.show()

        print("✅ 종합 대시보드 생성 완료")

    except Exception as e:
        print(f"⚠️ 대시보드 생성 실패: {e}")


# 7. 시스템 실행
if __name__ == "__main__":
    # 고신뢰도 시스템 실행
    data, models, prediction = run_high_reliability_system()

    if prediction:
        print("\n🎉 고신뢰도 삼성전자 예측 시스템 실행 완료!")

        print("\n📌 시스템 특징:")
        print("  ✅ 최대 범위 데이터 수집 (최대 10년)")
        print("  ✅ 포괄적 시장 지표 (50+ 개)")
        print("  ✅ 고급 기술적 지표 (100+ 개)")
        print("  ✅ 다중 모델 앙상블 (10개 모델)")
        print("  ✅ 동적 가중치 시스템")
        print("  ✅ 신뢰구간 계산")
        print("  ✅ 피처 선택 자동화")
        print("  ✅ 교차 검증 기반 안정성 평가")

        # 추가 통계
        if data is not None:
            print(f"\n🔍 시스템 통계:")
            print(f"  📊 총 데이터 포인트: {len(data):,}개")
            print(f"  📅 데이터 기간: {(data.index.max() - data.index.min()).days:,}일")
            print(f"  🧬 총 피처 수: {len(data.columns):,}개")
            print(f"  📈 신뢰도 점수: {prediction.get('confidence_score', 0):.1f}/100")

            if models:
                best_model = max(models.keys(), key=lambda x: models[x]['val_r2'])
                print(f"  🏆 최고 성능 모델: {best_model} (R²: {models[best_model]['val_r2']:.4f})")

                avg_r2 = np.mean([m['val_r2'] for m in models.values()])
                print(f"  📊 평균 모델 성능: R² = {avg_r2:.4f}")
    else:
        print("❌ 시스템 실행 실패")


🚀 고신뢰도 삼성전자 예측 시스템 로딩 완료!
🚀 고신뢰도 삼성전자 예측 시스템 시작!
📊 1단계: 최대 범위 데이터 수집
📊 최대 범위 삼성전자 데이터 수집 중...
✅ 삼성전자 데이터: 6381일 (max) - 2000-01-03 15:00:00 ~ 2025-07-01 15:00:00
📈 33개 시장 지표 수집 중...
📊 진행률: 5/33
📊 진행률: 10/33


$CNYKRW=X: possibly delisted; no price data found  (1d 2000-01-03 15:00:00 -> 2025-07-01 15:00:00)


📊 진행률: 15/33
📊 진행률: 20/33
📊 진행률: 25/33


HTTP Error 404: 
$^VKOSPI: possibly delisted; no timezone found


📊 진행률: 30/33
✅ 시장 데이터 수집 완료: 31개
✅ 관련 회사 데이터: 13개
📈 수집된 데이터:
  - 주가 데이터: 6381일
  - 시장 지표: 31개
  - 관련 회사: 13개

📈 2단계: 고급 기술적 지표 계산
📈 고급 기술적 지표 계산 중...
✅ 고급 기술적 지표 계산 완료: (6381, 161)

📰 3단계: 뉴스 감성 분석
⚠️ Google 뉴스 (Samsung Electronics) 실패: URL can't contain control characters. '/rss/search?q=Samsung Electronics&hl=ko&gl=KR&ceid=KR:ko' (found at least ' ')
⚠️ Google 뉴스 (Samsung chip) 실패: URL can't contain control characters. '/rss/search?q=Samsung chip&hl=ko&gl=KR&ceid=KR:ko' (found at least ' ')
⚠️ Google 뉴스 (Samsung semiconductor) 실패: URL can't contain control characters. '/rss/search?q=Samsung semiconductor&hl=ko&gl=KR&ceid=KR:ko' (found at least ' ')
⚠️ Google 뉴스 (삼성 반도체) 실패: URL can't contain control characters. '/rss/search?q=%EC%82%BC%EC%84%B1 %EB%B0%98%EB%8F%84%EC%B2%B4&hl=ko&gl=KR&ceid=KR:ko' (found at least ' ')
✅ 뉴스 수집 완료: 36개
🧠 고급 감성 분석 수행 중...
✅ 고급 감성 분석 완료: 9일

🔧 4단계: 데이터 통합 및 피처 엔지니어링
🔧 데이터 통합 중...
✅ 데이터 통합 완료: (6381, 221)
🧬 고급 피처 엔지니어링 중...
🔧 결측치 처리 중...
✅ 고급 피처 엔지니어링 완료: (

Traceback (most recent call last):
  File "C:\Users\It\AppData\Local\Temp\ipykernel_6424\3104911410.py", line 1194, in run_high_reliability_system
    selected_features = prediction_system.feature_selection(X_train, y_train, feature_cols)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\It\AppData\Local\Temp\ipykernel_6424\3104911410.py", line 821, in feature_selection
    X_selected = selector_stats.fit_transform(X_train, y_train)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\It\anaconda3\Lib\site-packages\sklearn\utils\_set_output.py", line 313, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\It\anaconda3\Lib\site-packages\sklearn\base.py", line 1101, in fit_transform
    return self.fit(X, y, **fit_params).transform(X)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\It\anaconda3\Lib\site-packages\sklearn\bas